<h2>Start Visual NLP Session</h2>

In [1]:
from sparkocr import start
import math
import os
import json
import time
import shutil
import glob

os.environ['JAVA_HOME'] = '/home/linuxbrew/.linuxbrew/Cellar/openjdk@21/21.0.10'
license = "/workspace/spark_nlp_for_healthcare_spark_ocr_10538.json"

if license and "json" in license:

    with open(license, "r") as creds_in:
        creds = json.loads(creds_in.read())

        for key in creds.keys():
            os.environ[key] = creds[key]
else:
    raise Exception("License JSON File is not specified")

extra_configurations = {
    "spark.extraListeners": "com.johnsnowlabs.license.LicenseLifeCycleManager"
    "spark.sql.legacy.allowUntypedScalaUDF" : "true"
}

spark = start(secret=os.environ.get("SPARK_OCR_SECRET"),
              nlp_secret=os.environ.get("SECRET"),
              nlp_internal=os.environ.get("JSL_VERSION"),
              apple_silicon=False,
              nlp_version=os.environ.get("PUBLIC_VERSION"),
              logLevel="ERROR",
              use_gpu=True,
              extra_conf=extra_configurations)

spark

Spark version: 3.5.0
Spark NLP version: 6.2.0
Spark NLP for Healthcare version: 6.2.2
Spark OCR version: 6.3.0rc1

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4106152d-1664-4590-aaed-8fa80e11ec25;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp_2.12;6.2.2 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in central
	found com.amazonaws#jmespath-java;1.12.500 in central
	found com.g

<h2>Load Visual NLP Packages</h2>

In [2]:
from sparkocr.transformers import *
from sparkocr.enums import *
from sparkocr.utils import *
from pyspark.ml import PipelineModel, Pipeline

from sparknlp.annotator import *
from sparknlp.base import *
import sparknlp_jsl
from sparknlp_jsl.annotator import *

from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
import sqlite3
import json 

## Databricks File Mapping and Results

https://github.com/databricks-industry-solutions/pixels/blob/2fdbe9835015cc428473d7407ba01a623c407408/03b-Image-DeIdentification.ipynb

We perform lightweight parsing on the Pixels outputs to extract:
- Input file names
- Corresponding de-identification results

In [55]:
databricks_mapping = {
 '2.3.957.1.1.1204841.4.221.1244497244821555782__1-1.dcm': 'HAYES JESSICA, 01.03.2012, 12.15.1986',
 '3.4.868.1.1.3410519.2.594.1727310115838398258__1-2.dcm': '',
 '2.4.332.0.2.8029662.9.504.3152287158250332395__1-314.dcm': '',
 '1.2.692.1.2.8881663.7.630.2602716578066938264__1-0186.dcm': '',
 '3.1.874.1.3.8955936.3.406.2571635843887543751__1-1155.dcm': '',
 '3.3.887.0.0.7990059.2.604.6609992527253610405__1-02.dcm': '',
 '3.3.643.1.2.3791796.5.188.3075860933482453746__1-1.dcm': '',
 '3.2.963.0.1.1744807.5.560.2925915423073421345__1-48.dcm': '',
 '2.1.239.0.1.5573651.5.438.2234045507897739313__1-1.dcm': 'JOHNSON LARRY [M], 03.09.2019, 01.30.1968',
 '1.3.699.1.2.1566659.8.694.3677882864699772127__1-1.dcm': '',
 '3.1.755.1.3.3756213.9.632.2823904128812494525__1-1245.dcm': '',
 '1.5.957.1.0.0469346.1.591.1423388835865443258__1-305.dcm': '',
 '2.4.167.1.0.5495280.2.998.2172758132580859656__1-527.dcm': '',
 '2.4.569.0.1.8829737.8.747.1088525486356273997__1-1.dcm': 'JONES ERIN [F], 01.12.2019, 11.12.1937',
 '3.4.769.1.3.3147177.1.328.1201733738289342904__1-2.dcm': '',
 '2.2.770.1.0.9549343.7.515.1683676004402516592__1-1401.dcm': '',
 '1.4.229.0.1.8882135.4.232.9368788976563694449__1-1.dcm': '',
 '1.1.740.0.2.4463594.1.836.8941495612734419636__1-1.dcm': '',
 '3.1.755.1.3.3756213.9.632.3912467996779888855__1-1474.dcm': '',
 '1.2.877.0.2.4801540.7.178.1146080478315419110__1-1.dcm': '',
 '3.1.874.1.3.8955936.3.406.2571635843887543751__1-1118.dcm': '',
 '3.4.732.1.3.0861594.8.259.2531881142306101964__1-1.dcm': 'MOODY BRENT, 08.06.2010, 06.01.1922',
 '3.1.755.1.3.3756213.9.632.2823904128812494525__1-1950.dcm': '',
 '2.2.770.1.0.9549343.7.515.1683676004402516592__1-0881.dcm': '',
 '2.3.625.0.1.8002214.6.071.3116750353259316847__1-2.dcm': '',
 '3.2.435.1.0.5381938.1.237.9592273895273147042__1-2.dcm': '',
 '3.3.887.0.0.7990059.2.604.6609992527253610405__1-29.dcm': '',
 '1.1.916.0.2.2375140.9.390.3317408698816973276__1-20.dcm': '',
 '2.3.568.1.0.7053673.2.382.1770839949180489836__1-1.dcm': 'BELL AUSTIN, 02.29.2012, 01.24.1964',
 '3.3.608.1.1.2993058.8.105.1343059462659162280__1-1.dcm': 'GARCIA MEAGAN, 03.30.2013, 02.24.1967',
 '3.3.120.1.3.7921122.9.635.9581167083853861672__1-1.dcm': '',
 '3.4.250.0.3.0447726.8.787.1295625342042406287__1-313.dcm': '',
 '3.1.874.1.3.8955936.3.406.2571635843887543751__1-1252.dcm': '',
 '2.3.625.0.1.8002214.6.071.3116750353259316847__1-1.dcm': '',
 '3.3.718.1.2.8982237.8.011.5174446821187527015__1-1.dcm': '',
 '3.2.969.0.1.4856258.2.740.3049947016622309734__1-03.dcm': '',
 '2.4.823.0.0.4329134.1.917.2393059339015278965__1-106.dcm': '',
 '3.4.585.0.0.8284097.9.501.1675956140590704039__1-2053.dcm': '',
 '3.3.644.0.0.5312286.5.579.1650059363517082486__1-1.dcm': '',
 '1.3.241.0.0.3328322.7.935.9222594282780660977__1-029.dcm': '',
 '2.1.497.0.0.1835643.9.081.1307210876115091140__1-1.dcm': 'JAMES RUSSELL, 12.13.2018, 10.21.1947',
 '1.5.143.0.1.9349672.6.822.1258491400848416678__1-60.dcm': '',
 '1.4.249.1.1.0462139.1.347.9280022882387155784__1-1.dcm': '',
 '3.5.481.0.2.6038271.6.920.2933890315408309625__1-18.dcm': '',
 '2.4.357.1.0.9259302.8.497.3349013607548370215__1-70.dcm': '',
 '1.5.874.0.0.2194661.8.142.1156005950586769605__1-1.dcm': '',
 '3.1.142.1.0.8336226.8.709.1198851118360491096__1-1.dcm': '',
 '1.2.398.1.2.6768134.2.774.2820592714662272665__1-1.dcm': 'ADAMS LAURA, 10.14.2010, 08.31.1952',
 '3.1.755.1.3.3756213.9.632.2823904128812494525__1-1119.dcm': '',
 '1.2.967.0.0.6572422.4.113.1794077009491151390__1-1.dcm': '',
 '2.1.656.0.2.8048482.9.537.1658163530109825238__1-1.dcm': '',
 '1.3.150.0.2.6162827.2.658.1521624518310452739__1-1.dcm': '',
 '3.4.585.0.0.8284097.9.501.1675956140590704039__1-0288.dcm': '',
 '3.4.769.1.3.3147177.1.328.1201733738289342904__1-1.dcm': '',
 '2.2.770.1.0.9549343.7.515.1683676004402516592__1-0136.dcm': '',
 '3.1.874.1.3.8955936.3.406.2571635843887543751__1-0675.dcm': '',
 '3.1.755.1.3.3756213.9.632.3912467996779888855__1-0025.dcm': '',
 '1.5.566.0.3.5921028.1.523.5821238071079767946__1-1.dcm': 'SANCHEZ TIMOTHY, 08-05-2018, 07-01-1972',
 '3.4.868.1.1.3410519.2.594.1727310115838398258__1-1.dcm': '',
 '3.4.386.0.1.9799157.3.927.2629160371996664589__1-35.dcm': '',
 '2.2.311.1.3.2905093.6.827.3459617097241474781__1-1.dcm': '',
 '3.4.250.0.3.0447726.8.787.1295625342042406287__1-374.dcm': '',
 '1.5.612.0.2.9474539.7.488.8142452478919134598__1-1.dcm': '',
 '3.2.435.1.0.5381938.1.237.9592273895273147042__1-1.dcm': '',
 '1.4.861.0.2.1161202.9.705.1343320574652553137__1-88.dcm': '',
 '3.1.232.0.0.3150771.5.288.1050421815979028971__1-1.dcm': '',
 '2.4.674.0.0.2358241.5.296.3361166386730502735__1-1.dcm': 'MARTIN ERIC [M], 09.20.2010, 07.28.1938',
 '1.2.692.1.2.8881663.7.630.2602716578066938264__1-1233.dcm': '',
 '2.2.770.1.0.9549343.7.515.1683676004402516592__1-0694.dcm': '',
 '3.4.585.0.0.8284097.9.501.1675956140590704039__1-0080.dcm': ''
}

<h2>Find All Validation Files</h2>

In [56]:
midib_validation_root_path = "/workspace/MIDI-B/data/TCIA-MIDI-B-Synthetic-Validation_20250502/**/*.dcm"

dcm_files = glob.glob(midib_validation_root_path, recursive=True)

<h2>Filtering Validation Files Used in the Databricks Pixels Results</h2>

In [58]:
local_path_to_db = {}

for file in dcm_files:
    like_db = "__".join(file.split("/")[-2:])
    if like_db in databricks_mapping.keys():
        local_path_to_db[file] = like_db

print(f"Total Files found in MIDI-B Validation Set : {len(dcm_files)}")
print(f"Total Files in Databricks Pixels Result : {len(databricks_mapping.keys())}")
print(f"Total files in the MIDI-B validation set matching the Pixels platform: {len(local_path_to_db)}")

Total Files found in MIDI-B Validation Set : 23921
Total Files in Databricks Pixels Result : 70
Total files in the MIDI-B validation set matching the Pixels platform: 70


<h2>Define Pixel De-Identification Pipeline</h2>

In [ ]:
# Pre MIDI-B DeIdentifcation Pipeline

from sparknlp.pretrained import PretrainedPipeline
deid_pipeline = PretrainedPipeline("clinical_deidentification_docwise_benchmark_large", "en", "clinical/models")

dicom_to_metadata = DicomToMetadata() \
    .setInputCol("content") \
    .setOutputCol("metadata") \
    .setKeepInput(True)

dicom_to_image = DicomToImageV3() \
    .setInputCols(["content"]) \
    .setOutputCol("image_raw") \
    .setKeepInput(False)

text_detector = ImageTextDetector.pretrained("image_text_detector_mem_opt", "en", "clinical/ocr") \
    .setInputCol("image_raw") \
    .setOutputCol("text_regions") \
    .setScoreThreshold(0.7) \
    .setWithRefiner(True) \
    .setUseGPU(True) \
    .setWidth(0)

ocr = ImageToTextV2.pretrained("ocr_large_printed_v2_opt", "en", "clinical/ocr") \
    .setRegionsColumn("text_regions") \
    .setInputCols(["image_raw"]) \
    .setOutputCol("text") \
    .setOutputFormat("text_with_positions") \
    .setGroupImages(False) \
    .setKeepInput(False) \
    .setUseGPU(True) \
    .setUseCaching(True) \
    .setBatchSize(4)

regex_matcher = RegexMatcher()\
    .setInputCols("document")\
    .setOutputCol("regex")\
    .setRules([
        r"(?:\s[MFU]|\b[MFU])(?:\s|\b|$);GENDER",
        r"\b(?:JT|SWU|JKR|MWF|ICG|NKF|YH|TJN|LEITO|ACO|CEF|CMS|JGR|MSS|MHS|ROC|LM|RCN|FTA|MGO|LACI|VV|HA|TR|CJA)\b;CODE"]) \
    .setDelimiter(";")

chunkConverter = ChunkConverter()\
    .setInputCols("regex")\
    .setOutputCol("regex_chunks")

chunk_merger = ChunkMergeApproach()\
    .setInputCols('regex_chunks', "ner_chunk")\
    .setOutputCol('merged_ner_chunk')\
    .setMergeOverlapping(True)

chunkerFilter = ChunkFilterer() \
  .setInputCols(["document", "merged_ner_chunk"]) \
  .setOutputCol("filtered_chunks") \
  .setCriteria("isin") \
  .setBlackList(["@", "O", "6", "G", "e"])

position_finder = PositionFinder() \
    .setInputCols("filtered_chunks") \
    .setOutputCol("coordinates") \
    .setPageMatrixCol("positions") \
    .setSmoothCoordinates(True)

stages = deid_pipeline.model.stages[:-2].copy()
stages.insert(0, dicom_to_metadata)
stages.insert(1, dicom_to_image)
stages.insert(2, text_detector)
stages.insert(3, ocr)
stages.append(regex_matcher)
stages.append(chunkConverter)
stages.append(chunk_merger)
stages.append(chunkerFilter)
stages.append(position_finder)

pre_midib_pipeline = Pipeline(stages=stages)

In [ ]:
# Post MIDI-B DeIdentifcation Pipeline
dicom_to_image = DicomToImageV3() \
    .setInputCols(["content"]) \
    .setOutputCol("image_raw") \
    .setKeepInput(False)

text_detector = ImageTextDetector.pretrained("image_text_detector_mem_opt", "en", "clinical/ocr") \
    .setInputCol("image_raw") \
    .setOutputCol("text_regions") \
    .setScoreThreshold(0.7) \
    .setWithRefiner(True) \
    .setUseGPU(False) \
    .setWidth(0)

ocr = ImageToTextV3() \
    .setInputCols(["image_raw", "text_regions"]) \
    .setOutputCol("text")

p_document_assembler = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("p_document") \
    .setCleanupMode("disabled")
    
p_sentencer = SentenceDetector()\
    .setInputCols(["p_document"])\
    .setOutputCol("p_sentence") \
    .setCustomBounds(["\n"]) \
    .setUseCustomBoundsOnly(False)

p_tokenizer = Tokenizer() \
    .setInputCols(["p_sentence"]) \
    .setOutputCol("p_token")

labels = ["DATE", "DOCTOR", "PATIENT"]
p_zeroshot_ner_deid_subentity_docwise_medium = PretrainedZeroShotNER().pretrained("zeroshot_ner_deid_subentity_docwise_medium", "en", "clinical/models")\
    .setInputCols("p_sentence", "p_token")\
    .setOutputCol("p_zeroshot_ner_deid_subentity_docwise_medium")\
    .setPredictionThreshold(0.5)\
    .setLabels(labels)

p_zeroshot_ner_deid_subentity_docwise_medium_ner_converter = NerConverterInternal()\
    .setInputCols("p_sentence", "p_token", "p_zeroshot_ner_deid_subentity_docwise_medium")\
    .setOutputCol("p_zeroshot_ner_deid_subentity_docwise_medium_ner_chunk") \
    .setThreshold(0.80)

p_regex_matcher = RegexMatcher()\
    .setInputCols("p_document")\
    .setOutputCol("p_regex")\
    .setRules([
        r"\[\s*([MFU])\s*\](?:\s|\b|$);GENDER",
        r"\b(?:JT|SWU|JKR|MWF|ICG|NKF|YH|TJN|LEITO|ACO|CEF|CMS|JGR|MSS|MHS|ROC|LM|RCN|FTA|MGO|LACI|VV|HA|TR|CJA)\b;CODE"]) \
    .setDelimiter(";")

p_chunk_converter = ChunkConverter()\
    .setInputCols("p_regex")\
    .setOutputCol("p_regex_chunk")

p_chunk_merger = ChunkMergeApproach()\
    .setInputCols('p_regex_chunk', "p_zeroshot_ner_deid_subentity_docwise_medium_ner_chunk")\
    .setOutputCol('p_merged_ner_chunk')\
    .setMergeOverlapping(True)


post_midib_pipeline = Pipeline(stages=[
    dicom_to_image,
    text_detector,
    ocr,
    p_document_assembler,
    p_sentencer,
    p_tokenizer,
    p_zeroshot_ner_deid_subentity_docwise_medium,
    p_zeroshot_ner_deid_subentity_docwise_medium_ner_converter,
    p_regex_matcher,
    p_chunk_converter,
    p_chunk_merger
])

image_text_detector_mem_opt download started this may take some time.
Approximate size to download 77.5 MB
zeroshot_ner_deid_subentity_docwise_medium download started this may take some time.
Approximate size to download 678.7 MB
[OK!]


<h2>Run pipeline and checkpoint result to disk</h2>

In [ ]:
db_files = list(local_path_to_db.keys())
batched = False

if batched:
    # If running in standalone mode
    batch_size = 2
    
    for i in range(0, len(db_files), batch_size):
        batch_files = db_files[i:i + batch_size]
        df = spark.read.format("binaryFile").load(batch_files)
        result = post_midib_pipeline.fit(df).transform(df)
        result.write.format("parquet").mode("append").save(f"./result_visual_nlp_db_samples")

else:

    df = spark.read.format("binaryFile").load(db_files)
    result = post_midib_pipeline.fit(df).transform(df)
    result.write.format("parquet").mode("overwrite").save(f"./result_visual_nlp_db_samples")

<h2>Load Result from disk</h2>

In [59]:
input_df = spark.read.format("parquet").load(f"./result_visual_nlp_db_samples")
input_df.columns

['pagenum',
 'frame_dims',
 'path',
 'modificationTime',
 'length',
 'text_regions',
 'text',
 'confidence',
 'positions',
 'exception',
 'p_document',
 'p_sentence',
 'p_token',
 'p_zeroshot_ner_deid_subentity_docwise_medium',
 'p_zeroshot_ner_deid_subentity_docwise_medium_ner_chunk',
 'p_regex',
 'p_regex_chunk',
 'p_merged_ner_chunk']

In [63]:
jsl_result = {}

for row in input_df.select("path", "p_merged_ner_chunk").toLocalIterator():
    data = row.asDict()
    basepath = "__".join(data["path"].split("/")[-2:])
    
    text = []
    for ner in data["p_merged_ner_chunk"]:
        text.append(ner.result)

    jsl_result[basepath] = " ".join(text)

In [64]:
jsl_result

{'1.5.566.0.3.5921028.1.523.5821238071079767946__1-1.dcm': 'SANCHEZ TIMOTHY [M]  08.05.2018 07.01.1972',
 '2.1.497.0.0.1835643.9.081.1307210876115091140__1-1.dcm': 'JAMES RUSSELL [M]  12.13.2018 10.21.1947',
 '2.4.674.0.0.2358241.5.296.3361166386730502735__1-1.dcm': 'MARTIN ERIC [M]  09.20.2010 07.28.1938',
 '2.3.957.1.1.1204841.4.221.1244497244821555782__1-1.dcm': 'HAYES JESSICA [F]  01.03.2012 12.15.1986',
 '2.1.239.0.1.5573651.5.438.2234045507897739313__1-1.dcm': 'JOHNSON LARRY [M]  03.09.2019 01.30.1968',
 '3.3.608.1.1.2993058.8.105.1343059462659162280__1-1.dcm': 'GARCIA MEAGAN [F]  03.30.2013 02.24.1967',
 '1.2.398.1.2.6768134.2.774.2820592714662272665__1-1.dcm': 'ADAMS LAURA [U]  10.14.2010 08.31.1952',
 '2.3.568.1.0.7053673.2.382.1770839949180489836__1-1.dcm': 'BELL AUSTIN [U]  02.29.2012 01.24.1964',
 '2.4.569.0.1.8829737.8.747.1088525486356273997__1-1.dcm': 'JONES ERIN [F]  01.12.2019 11.12.1937',
 '3.4.732.1.3.0861594.8.259.2531881142306101964__1-1.dcm': 'MOODY BRENT [M]  08.

<h2>Generate Combined Result</h2>

Here, we read the validation database to identify files marked for **pixel hiding actions**.  
This step determines the complete set of files that require pixel-level masking.

We then generate the **`SeriesInstanceUID_Validation_Collection`**, which serves as the **ground truth** indicating all files that contain PHI within the image pixels.

Next, we compare the outputs from:
- **John Snow Labs Visual NLP (`jsl_result`)**
- **Databricks Pixels Platform (`databricks_mapping`)**
- **`SeriesInstanceUID_Validation_Collection` (ground truth)**

Based on this comparison, we generate a consolidated validation result with the following fields:

- **Validation File** → File name  
- **Contains PHI** → Whether the file key is present in `SeriesInstanceUID_Validation_Collection` ( boolean )  
  *(i.e., whether MIDI-B marks the file as requiring pixel-level cleaning)*  
- **Ground Truth** → Presence of PHI in pixel data according to MIDI-B  
- **JSL Prediction** → PHI predicted by Visual NLP ( boolean )
- **JSL Detected** → Whether Visual NLP detects any PHI in the file  
- **DB Prediction** → PHI predicted by the Databricks Pixels platform ( boolean )
- **DB Detected** → Whether the Databricks Pixels platform detects any PHI in the file

In [14]:
validation_db_path = "/workspace/MIDI-B/metadata/MIDI-B-Answer-Key-Validation.db"

conn = sqlite3.connect(validation_db_path)
cursor = conn.cursor()

cursor.execute("SELECT * FROM answer_data")
rows = cursor.fetchall()

In [40]:
SeriesInstanceUID_Validation_Collection = {}

for row in rows:    
    data = json.loads(row[9])
    
    for key, item in data.items():
        if item["action"] == "<pixels_hidden>":
            temp = json.loads(item["action_text"].replace("<", "").replace(">", ""))
            temp["contains_phi"] = 0
            SeriesInstanceUID_Validation_Collection[str(row[6])] = temp

In [43]:
results = []

for key in databricks_mapping.keys():
    jsl_text = jsl_result[key].replace("\n", " ") if jsl_result[key].strip() != "" else None
    db_text = databricks_mapping[key] if databricks_mapping[key].strip() != "" else None

    if db_text == None:
        db_redacted = 0
    else:
        db_redacted = 1

    if jsl_text == None:
        jsl_redacted = 0
    else:
        jsl_redacted = 1
        
    try:
        gt_text = SeriesInstanceUID_Validation_Collection[key.split("__")[0]]["text"].replace("\n", " ")
        contains_phi = 1
    except Exception as E:
        gt_text = None
        contains_phi = 0 
        
    results.append([key.split("__")[0], contains_phi, gt_text, jsl_text, jsl_redacted, db_text, db_redacted])

In [44]:
df = pd.DataFrame(results, columns=["Validation File", "Contains PHI", "Ground Truth", 
                                    "JSL Prediction", "JSL Detected", "DB Prediction", "DB Detected"])

In [50]:
df.head()

,Validation File,Contains PHI,Ground Truth,JSL Prediction,JSL Detected,DB Prediction,DB Detected
0,2.3.957.1.1.1204841.4.221.1244497244821555782,1,HAYES JESSICA [F] 01.03.2012 DOB: 12.15.1986,HAYES JESSICA [F] 01.03.2012 12.15.1986,1,"HAYES JESSICA, 01.03.2012, 12.15.1986",1
1,3.4.868.1.1.3410519.2.594.1727310115838398258,1,ICG,ICG,1,NaN,0
2,2.4.332.0.2.8029662.9.504.3152287158250332395,0,NaN,NaN,0,NaN,0
3,1.2.692.1.2.8881663.7.630.2602716578066938264,0,NaN,NaN,0,NaN,0
4,3.1.874.1.3.8955936.3.406.2571635843887543751,0,NaN,NaN,0,NaN,0


In [ ]:
df.to_excel("./jsl_vs_db_midib_pipeline_results.xlsx", index=False)

<h2>JSL Metrics</h2>

In [48]:
y_true = df["Contains PHI"]
y_pred = df["JSL Detected"]

precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print(f"Precision : {precision} Recall : {recall}, F1-Score : {f1}")

JSL METRICS
Precision : 1.0 Recall : 0.7142857142857143, F1-Score : 0.8333333333333334


<h2>Databricks Pixels Metrics</h2>

In [62]:
y_true = df["Contains PHI"]
y_pred = df["DB Detected"]

precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print(f"Precision : {precision} Recall : {recall}, F1-Score : {f1}")

Precision : 1.0 Recall : 0.2857142857142857, F1-Score : 0.4444444444444444
